In Lesson 1, we saw our raw LLM fail at a complex task because it suffered from Frozen Knowledge (no internet access) and Mathematical Constraints (inability to compute exact math).

To solve this manually, we would have to write hundreds of lines of complex boilerplate code to build API connectors, parse text, and create while loops.
Instead, we will use LangChain to orchestrate this for us. We will give our model two Tools:

Google Search: To browse the web for the current president's age.
Python REPL: A Python execution environment to accurately calculate the exact square root.

Then, we will use an Agent to dynamically break the question down. The Agent will enter a reasoning loop where it will Reason about what to do, Act by calling a tool, and Observe the result until it reaches the final synthesized answer.

In [ ]:
# 1. Install the necessary LangChain packages
# Note: We include langchain-experimental because it contains the PythonREPLTool
!pip install -U --quiet langchain langchain_core langchain_community langchain_openai langchain_experimental langchainhub
!pip install -U langchain-tavily
import warnings

# Suppress all DeprecationWarnings globally in Colab
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
from openai import OpenAI
from google.colab import userdata
api_key=userdata.get('OPENAI_API_KEY')

Building the Agent
Now we will hand control over to LangChain. We will define our tools, pull a pre-tested prompt from the LangChain Hub (a community repository for prompts), and let the AgentExecutor do the heavy lifting of running the reasoning loop.

⚠️ Security Note: We are using PythonREPLTool for this mathematical calculation. This tool executes arbitrary Python code under the hood. While it is excellent for development and exact math, you should be careful using it in production environments without proper sandboxing!

In [ ]:
# Import the latest agent creation utility
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from google.colab import userdata
from langchain_tavily import TavilySearch
import os
# 1. Initialize the LLM
# Note: ensure you have OPENAI_API_KEY in your Colab secrets
api_key = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
llm = ChatOpenAI(api_key=api_key, model="gpt-4o-mini", temperature=0)

In [ ]:
import langchain
print(f"LangChain version: {langchain.__version__}")

LangChain version: 1.2.15


In [ ]:
# 2. Define the tools LangChain can hand over to the LLM

# You need to set up Tavily Search API Key

searchtool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # start_date=None,
    # end_date=None,
    # include_domains=None,
    # exclude_domains=None,
    # include_usage= False
)

tools = [
   searchtool, # Tool for finding the current president and his age
    PythonREPLTool()   # Tool for calculating the exact square root
]

In [ ]:
# 3. Create the agent with a more explicit step-by-step prompt
agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are a helpful assistant.
To solve the user's request, follow these steps exactly:
1. Use Tavily Search to identify the current President of the United States.
2. Use Tavily Search to find that specific person's current age.
3. Use the Python REPL to calculate the square root of that age.
4. Provide the final answer with all the details.

Do not try to guess. Use your tools sequentially."""
)

In [ ]:
# 4. Execute the updated query
inputs = {"messages": [{"role": "user", "content": "Who is the current US president and what is the square root of their age multiplied by 132?"}]}

# Using stream to see the reasoning steps in real-time
for chunk in agent_executor.stream(inputs, stream_mode="values"):
    message = chunk["messages"][-1]
    message.pretty_print()

================================ Human Message =================================

Who is the current US president and what is the square root of their age multiplied by 132?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_d4zMCi2DDFzyyKSiSHokLPXS)
 Call ID: call_d4zMCi2DDFzyyKSiSHokLPXS
  Args:
    query: current President of the United States
================================= Tool Message =================================
Name: tavily_search

{"query": "current President of the United States", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.usa.gov/presidents", "title": "Presidents, vice presidents, and first ladies - USAGov", "content": "Current president. The 47th and current president of the United States is Donald John Trump. He was sworn into office on January 20, 2025", "score": 0.9940428, "raw_content": null}, {"url": "https://en.wikipedia.org/wiki/President_of_the_Uni